In [ ]:
import os
import pickle
import json
import sys
import numpy as np

sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/utls/')
sys.path.append('/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/src/model/')
sys.path.append("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/")


from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from encoders import *
from utls.plotting import ensure_dir
from utls.loading import load_results_with_exclusion_2, move_sequences_to_used, load_results_with_exclusion_no_dropping
from utls.runners import run_experiment_scores
from utls.runners_v2 import (
    run_experiment_grid,
    run_experiment_scores,
    run_experiment_scores_itemwise,
    run_experiment_itemwise_hits_fas,
    make_noise_schedule
)

from utls.analysis_helpers import rocs_across_noise, convert_human_to_model_struct, compute_scaling_vs_human, convert_human_to_model_struct
from utls.analysis_helpers import auroc_to_dprime, compute_model_dprime_curve
from utls.analysis_helpers import roc_for_isi, auroc_to_dprime
from utls.plotting import plot_across_noise, plot_noise_overlays
from utls.io_utils import make_model_save_dir, save_all_figures, save_single_figure, save_runs_summary
from utls.roc_utils import roc_from_arrays 
from utls.runners_utils import *

def load_all_bestmodel_pkls(fits_dir):
    """
    Load bestmodels-*.pkl files into a flat list of model records.
    Each pkl is assumed to contain exactly one best-fit model.
    """
    records = []

    for fname in sorted(os.listdir(fits_dir)):
        if not fname.endswith(".pkl"):
            continue

        path = os.path.join(fits_dir, fname)
        print(f"Loading {fname}")

        with open(path, "rb") as f:
            rec = pickle.load(f)

        # ---- sanity checks ----
        required_keys = {"params", "run_out", "nmse"}
        missing = required_keys - rec.keys()
        if missing:
            raise ValueError(f"{fname} missing keys: {missing}")

        params = rec["params"]

        record = {
            # core outputs
            "params": params,
            "run_out": rec["run_out"],
            "model_dp": rec.get("model_dp"),
            "nmse": rec["nmse"],

            # identity (prefer top-level, fall back to params)
            "encoder_name": rec.get("encoder") or params.get("encoder"),
            "layer": rec.get("layer") or params.get("layer"),
            "metric": params.get("metric"),
            "noise_mode": params.get("noise_mode"),
            "stimulus_set": rec.get("stimulus_set") or params.get("stimulus_set"),

            # provenance
            "source_file": fname,
        }

        records.append(record)

    return records

def inspect_bestmodel_pkl(path, max_keys=10):
    import pickle

    with open(path, "rb") as f:
        obj = pickle.load(f)

    print(f"\nFILE: {path}")
    print(f"Type: {type(obj)}")

    if isinstance(obj, list):
        print(f"List length: {len(obj)}")
        if len(obj) > 0 and isinstance(obj[0], dict):
            print("Keys in first element:")
            for k in list(obj[0].keys())[:max_keys]:
                print(f"  - {k}")

    elif isinstance(obj, dict):
        print(f"Dict keys (showing up to {max_keys}):")
        for k in list(obj.keys())[:max_keys]:
            print(f"  - {k}")
        first_val = next(iter(obj.values()))
        print("Value type:", type(first_val))
        if isinstance(first_val, dict):
            print("Keys in value:")
            for k in list(first_val.keys())[:max_keys]:
                print(f"  - {k}")

    else:
        print("Unknown object structure")

    return obj

from collections import defaultdict
def group_by_key(records, key):
    grouped = defaultdict(list)
    for rec in records:
        grouped[rec[key]].append(rec)
    return grouped


# def evaluate_grid_results(grid_results, human_curve, isis):
#     """
#     Compute d′ curve and NMSE for each model entry in grid_results.
#     Accepts either:
#       - rec["results"]  (legacy)
#       - rec["run_out"]  (post-hoc / bestmodel)
#     """
#     for rec in grid_results:
#         run_out = rec.get("results", rec.get("run_out"))

#         if run_out is None:
#             raise KeyError(
#                 "Record must contain 'results' or 'run_out'"
#             )

#         model_dp = compute_model_dprime_for_run(run_out, isis)
#         rec["model_dp"] = model_dp
#         rec["nmse"] = compute_nmse(model_dp, human_curve)

tasks = {
    0: "ind-nature",
    1: "global-music",
    2: "atexts",
}

In [ ]:
# ---- load model registry ----
fits_dir = "/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/fits/fits_v11_three-regime_cosine_median_dist_all"
results_root = "/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/"
all_opt_results = load_all_bestmodel_pkls(fits_dir)

print(f"Loaded {len(all_opt_results)} models")

# ---- enrich records (IMPORTANT) ----
for rec in all_opt_results:
    rec["t_step"] = rec["params"].get("t_step", None)

# ---- grouping (THIS IS WHERE YOUR QUESTION POINTS) ----
models_by_noise = group_by_key(all_opt_results, "noise_mode")

models_by_noise_task_repr = {}
for noise_mode, noise_recs in models_by_noise.items():
    by_task = group_by_key(noise_recs, "stimulus_set")

    models_by_noise_task_repr[noise_mode] = {}
    for task, task_recs in by_task.items():
        by_repr = group_by_key(task_recs, "encoder_name")
        models_by_noise_task_repr[noise_mode][task] = by_repr

models_by_noise_task_repr_t = {}

for noise_mode, task_dict in models_by_noise_task_repr.items():
    models_by_noise_task_repr_t[noise_mode] = {}

    for task, repr_dict in task_dict.items():
        models_by_noise_task_repr_t[noise_mode][task] = {}

        for enc, enc_recs in repr_dict.items():
            if noise_mode == "two-regime":
                by_t = group_by_key(enc_recs, "t_step")
            else:
                by_t = {None: enc_recs}

            models_by_noise_task_repr_t[noise_mode][task][enc] = by_t

In [ ]:
#models_by_task = group_by_task(all_opt_results)
isis = [0, 1, 2, 4, 8, 16, 32, 64]
is_multi = True
which_isi = None

which_tasks = [0,1,2]

human_dprime_curve = {}

for which_task, task_label in tasks.items():
    exp_list, all_files, name_to_idx, human_runs, task_name, hr_task_name = \
        load_experiment_data(which_task, which_isi, is_multi, old=False)

    human_curve = compute_human_curve(human_runs, is_multi, which_isi)
    human_dprime_curve[hr_task_name] = human_curve

In [ ]:
from matplotlib.cm import get_cmap

def plot_best_models(
    best_models,
    human_curve,
    isis,
    hr_task_name="",
    encoder_name="",
    best_key="nmse"
):
    """
    Plot best models with:
      - color = representation
      - alpha = NMSE (better = darker)
      - linestyle = noise mode
    """

    plt.figure(figsize=(13, 9))

    # -------------------------
    # Human reference
    # -------------------------
    plt.plot(
        isis,
        human_curve,
        "o-k",
        linewidth=3,
        label="Human",
        zorder=10,
    )

    # -------------------------
    # Collect metadata
    # -------------------------
    nmses = np.array([rec[best_key] for rec in best_models.values()])
    nmse_min, nmse_max = np.nanmin(nmses), np.nanmax(nmses)

    def nmse_to_alpha(nmse):
        # best → alpha≈1, worst → alpha≈0.3
        if nmse_max == nmse_min:
            return 0.8
        return 0.3 + 0.7 * (nmse_max - nmse) / (nmse_max - nmse_min)

    # unique representations
    repr_keys = []
    for rec in best_models.values():
        enc = rec["params"].get("encoder", "unknown")
        layer = rec["params"].get("layer", None)
        repr_keys.append((enc, layer))

    repr_keys = sorted(set(repr_keys))

    # color map per representation
    cmap = get_cmap("tab10")
    repr_to_color = {
        rk: cmap(i % 10)
        for i, rk in enumerate(repr_keys)
    }

    # -------------------------
    # Plot models
    # -------------------------
    for (metric, noise_mode), rec in best_models.items():

        p = rec["params"]
        enc = p.get("encoder", "unknown")
        layer = p.get("layer", None)
        nmse = rec[best_key]

        color = repr_to_color[(enc, layer)]
        alpha = nmse_to_alpha(nmse)

        linestyle = "--" if noise_mode == "constant" else "-"

        label = (
            f" {noise_mode} | {best_key}={nmse:.3f}"
        )

        plt.plot(
            isis,
            rec["model_dp"],
            linestyle,
            color=color,
            alpha=alpha,
            linewidth=2,
            marker="o",
            label=label,
        )

    # -------------------------
    # Formatting
    # -------------------------
    plt.xlabel("ISI (s)")
    plt.ylabel("d′")
    plt.grid(True, alpha=0.3)
    plt.title(
        f"{hr_task_name}\n"
        f"Best Models"
    )

    plt.legend(
        fontsize=8,
        loc="center left",
        bbox_to_anchor=(1.02, 0.5),
        borderaxespad=0.0,
    )
    
    plt.tight_layout(rect=[0, 0, 0.78, 1])
    plt.show()

In [ ]:
sigma0 = []
mse = []

for key, rec in best_models.items():
    print(key)
    sigma0.append(rec["params"]["sigma0"])
    mse.append(rec["nmse"])

sigma0 = np.asarray(sigma0)
mse = np.asarray(mse)

plt.figure(figsize=(5, 4))
plt.scatter(sigma0, mse, alpha=0.6)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("sigma0")
plt.ylabel("Global MSE")
plt.title("Global MSE vs sigma0")
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
def get_best_model(records, human_curve, isis, best_key="nmse"):
    """
    Return the single best model (min NMSE) from records.
    """
    evaluate_grid_results(records, human_curve, isis)
    valid = [r for r in records if np.isfinite(r[best_key])]
    if not valid:
        return None
    return min(valid, key=lambda r: r[best_key])

def plot_best_constant_vs_two_regime_by_tstep(
    task_name,
    task_records,
    human_curve,
    isis,
    savedir=None,
    best_key="nmse"
):
    """
    One plot for one stimulus set:
      - best constant model
      - best two-regime model at each t_step
      - human curve
    """

    # ----------------------------
    # Split by noise mode
    # ----------------------------
    const_recs = [r for r in task_records
                  if r["params"]["noise_mode"] != "two-regime"]

    two_regime_recs = [r for r in task_records
                       if r["params"]["noise_mode"] == "two-regime"]

    # ----------------------------
    # Best CONSTANT model
    # ----------------------------
    best_const = get_best_model(const_recs, human_curve, isis, best_key)

    # ----------------------------
    # Best TWO-REGIME per t_step
    # ----------------------------
    best_two_by_t = {}
    by_t = group_by_key(two_regime_recs, "t_step")

    for t_step, recs in by_t.items():
        if t_step is None:
            continue
        best = get_best_model(recs, human_curve, isis, best_key)
        if best is not None:
            best_two_by_t[t_step] = best

    # ----------------------------
    # Plot
    # ----------------------------
    plt.figure(figsize=(6, 4))

    # Human
    plt.plot(isis, human_curve, "o-k", lw=3, label="Human")

    # Constant
    if best_const is not None:
        plt.plot(
            isis,
            best_const["model_dp"],
            "--",
            lw=2,
            label=f"Constant (best)"
        )

    # Two-regime curves
    for t_step in sorted(best_two_by_t):
        rec = best_two_by_t[t_step]
        plt.plot(
            isis,
            rec["model_dp"],
            "-o",
            alpha=0.8,
            label=f"Two-regime t={t_step}"
        )

    plt.xlabel("ISI")
    plt.ylabel("d′")
    plt.title(f"{task_name}: Best Constant vs Two-Regime")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)

    if savedir:
        ensure_dir(savedir)
        plt.savefig(
            f"{savedir}/best_constant_vs_two_regime_by_tstep.png",
            dpi=200
        )

    plt.show()

In [ ]:
isis = [0, 1, 2, 4, 8, 16, 32, 64]

# iterate by TASK only
for task_name in human_dprime_curve.keys():

    human_curve = human_dprime_curve.get(task_name)
    if human_curve is None:
        continue

    # collect ALL models for this task across noise modes
    task_records = []

    for noise_mode, task_dict in models_by_noise_task_repr.items():
        if task_name not in task_dict:
            continue

        repr_dict = task_dict[task_name]
        for recs in repr_dict.values():
            task_records.extend(recs)

    if not task_records:
        continue

    plot_best_constant_vs_two_regime_by_tstep(
        task_name=task_name,
        task_records=task_records,
        human_curve=human_curve,
        isis=isis,
        savedir=f"{results_root}/figures_v2/final/{task_name}",
    )

In [ ]:
task_records